## Exercícios

> Retirados de [learn-python: sqlalchemy_orm-questions](https://aviadr1.github.io/learn-advanced-python/11_db_access/exercise/sqlalchemy_orm-questions.html).

#### Q1.

Baixa e extraia o arquivo compactado com o banco de dados [Chinook database](https://www.sqlitetutorial.net/sqlite-sample-database/). Salve o arquivo `chinook.db` na mesma pasta deste script.
* Link para baixar: http://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip

<img width=500 src=https://www.sqlitetutorial.net/wp-content/uploads/2015/11/sqlite-sample-database-color.jpg>


#### Q2.

Leia o código e os comentários das células a seguir para entender como acessamos os modelos ORM de um banco já existente.

In [ ]:
from sqlalchemy import create_engine, text, MetaData
from sqlalchemy.orm import Session

engine = create_engine("sqlite+pysqlite:///chinook.db", echo=False)

### extrai as classes da base de dados Chinook
metadata = MetaData()
metadata.reflect(engine)

# O metadata tem informações sobre as tabelas
# que serão usadas para criar os modelos ORM
for table_name, table in metadata.tables.items():
    print(table_name)
    print(table.columns.keys())
    print(table.columns.items())
    print('-'*25)

### configura o objeto Base mapeando os modelos ORM das tabelas
from sqlalchemy.ext.automap import automap_base
Base = automap_base(metadata=metadata)
Base.prepare()

# o objeto Base tem os modelos ORM que podemos usar
# para manipular o banco de dados
print(Base.classes.items())

In [ ]:
# A seguir um exemplo de query na tabela Albums
# usamos o objeto Base para acessar cada modelo ORM.

session = Session(engine)
res = session.scalars(select(Base.classes.albums))
first_album = res.first()
print(first_album.AlbumId, first_album.Title)

#### Q3. 
Com base nos códigos anteriores realize as operações solicitadas nas células a seguir:


In [ ]:
from sqlalchemy import select

# Consulta
res = session.scalars(select(Base.classes.tracks).limit(3))

# Iteração
for track in res:
    print(track.TrackId, track.Name)

In [ ]:
from sqlalchemy import select
from sqlalchemy.orm import Session

tracks = Base.classes.tracks
albums = Base.classes.albums

stmt = (
    select(tracks.Name, albums.Title)
    .join(albums, tracks.AlbumId == albums.AlbumId)
    .limit(20)
)

with Session(engine) as session:
    result = session.execute(stmt)

    for name, title in result:
        print(f"{name} - {title}")

In [ ]:
from sqlalchemy import select
from sqlalchemy.orm import Session

invoice_items = Base.classes.invoice_items
tracks = Base.classes.tracks

stmt = (
    select(tracks.Name, invoice_items.Quantity)
    .join(tracks, invoice_items.TrackId == tracks.TrackId)
    .limit(10)
)

with Session(engine) as session:
    result = session.execute(stmt)

    for name, quantity in result:
        print(f"{name} - Quantidade: {quantity}")

In [ ]:
from sqlalchemy import select, func, desc
from sqlalchemy.orm import Session

invoice_items = Base.classes.invoice_items
tracks = Base.classes.tracks

stmt = (
    select(
        tracks.Name,
        func.sum(invoice_items.Quantity).label("total_vendas")
    )
    .join(tracks, invoice_items.TrackId == tracks.TrackId)
    .group_by(tracks.TrackId)
    .order_by(desc("total_vendas"))
    .limit(10)
)

with Session(engine) as session:
    result = session.execute(stmt)

    for name, total in result:
        print(f"{name} - Vendidas: {total}")

In [ ]:
from sqlalchemy import select, func, desc
from sqlalchemy.orm import Session

invoice_items = Base.classes.invoice_items
tracks = Base.classes.tracks
albums = Base.classes.albums
artists = Base.classes.artists

stmt = (
    select(
        artists.Name,
        func.sum(invoice_items.Quantity).label("total_vendas")
    )
    .join(tracks, invoice_items.TrackId == tracks.TrackId)
    .join(albums, tracks.AlbumId == albums.AlbumId)
    .join(artists, albums.ArtistId == artists.ArtistId)
    .group_by(artists.ArtistId)
    .order_by(desc("total_vendas"))
    .limit(10)
)

with Session(engine) as session:
    result = session.execute(stmt)

    for name, total in result:
        print(f"{name} - Vendidas: {total}")